# chess-vi — SFT trên Colab

Notebook này **chỉ gọi script** `chessvi.train.sft`. Không copy-paste logic vào cell:
logic nằm trong repo để test được và để Colab với local chạy đúng một thứ.

Runtime cần: **A100 / L4 / T4 GPU**. Colab hay ngắt giữa chừng nên script bật
`hub_strategy="checkpoint"` — cell cuối resume lại từ checkpoint.

Thứ tự chạy: `Mount` → `Cài đặt` → `Token` → `GPU` → `C1 + dịch (T5)` →
`Validate (T6)` → `Smoke test` → `Train`.
Đứt kết nối thì chạy lại 4 cell đầu rồi nhảy thẳng xuống cell **Resume**.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/chessvi"
!mkdir -p {DRIVE_ROOT}/outputs

## 2. Lấy repo và cài đặt

Sửa `REPO_URL` thành remote của bạn. Nếu đã clone repo vào Drive thì chỉ cần `cd`.

In [ ]:
REPO_URL = "https://github.com/trantrien1/ChessVi.git"
REPO_DIR = "/content/ChessVi"

import os

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull --ff-only || true

# Bỏ -q: -q nuốt mất chi tiết xung đột dependency, chỉ còn trơ
# "ResolutionImpossible" không biết package nào đá nhau.
!pip install -e ".[train,data]" 2>&1 | tail -30

# pip thất bại KHÔNG làm notebook dừng lại — không chốt ở đây thì mọi cell sau
# sẽ chết vì ModuleNotFoundError và che mất nguyên nhân thật.
!python -c "import chessvi; print('chessvi OK:', chessvi.__file__)" || echo ">>> CÀI ĐẶT HỎNG — DỪNG LẠI, ĐỪNG CHẠY CELL SAU"

## 3. Token Hugging Face

Lưu token trong **Colab Secrets** (biểu tượng chìa khoá, tên `HF_TOKEN`).
Không bao giờ dán token thẳng vào cell — notebook sẽ bị commit kèm token.

In [ ]:
import os

# Colab Secrets chỉ đọc được khi chạy từ giao diện web colab.research.google.com:
# userdata.get() hỏi ngược về frontend trong trình duyệt. Chạy từ VS Code hay một
# frontend khác thì không ai trả lời -> TimeoutException. Bắt rộng vì mỗi trường
# hợp hỏng một kiểu (ImportError, TimeoutException, SecretNotFoundError).
if not os.environ.get("HF_TOKEN"):
    try:
        from google.colab import userdata

        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
    except Exception as error:
        print(f"Không lấy được Colab Secret ({type(error).__name__}).")
        import getpass

        # getpass không in token ra output, nên notebook commit lên không lộ.
        os.environ["HF_TOKEN"] = getpass.getpass("HF_TOKEN (Enter để bỏ qua): ")

# Token CHỈ cần khi push lên Hub (cell 7). Các cell validate/smoke test không cần.
if not os.environ.get("HF_TOKEN"):
    print("CHƯA CÓ HF_TOKEN — validate và smoke test vẫn chạy, cell 7 sẽ không push được.")

## 4. Kiểm tra GPU

In [ ]:
!nvidia-smi

import torch

print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")

## 5. Lấy và dịch dữ liệu C1 (T5)

`chessvi.data.c1` chuẩn hoá `UofTCSSLab/C1-data` về schema pipeline hiểu: tách
FEN ra khỏi câu văn thành cột riêng, rút nước sau `FINAL_ANSWER:` thành `label`.
Bỏ bước này thì T6 loại 100% mẫu với lý do `missing_fen`.

Backend dịch là **`llm`** (mặc định `Qwen/Qwen3-14B` qua vLLM), không phải máy
dịch phổ thông. Lứa dịch đầu bằng `envit5` cho ra `rook` = "cổ tay",
`checkmate` = "giao phối", và 75% mẫu mất luôn dòng kết luận. LLM được đưa
thẳng bảng thuật ngữ trong prompt và bị kiểm lại placeholder sau khi sinh.

Đổi model bằng `--model`. Qwen3-14B bf16 ~28GB, vừa A100 40GB; 32B và 30B-A3B
ở bf16 đều tràn.

Dịch 39.601 mẫu ước chừng **2–4 giờ A100** (~20–60 CU). **Luôn chạy `--dry-run`
trước** và đọc tận mắt 20 cặp before/after — sai ở đây thì toàn bộ nhãn thành
rác, và bạn chỉ phát hiện sau khi đã đốt hết CU.

In [ ]:
RAW = f"{DRIVE_ROOT}/data/raw/c1_sft.jsonl"
TRANSLATED = f"{DRIVE_ROOT}/data/translated/sft"

# Backend llm chạy bằng vLLM — chưa cài ở cell 2 nên cài ở đây.
!pip install -e ".[rl]" 2>&1 | tail -5

# Chuẩn hoá C1-data -> {id, fen, question, answer, label}. --limit để thử trước.
!python -m chessvi.data.c1 --split sft --limit 200 --out {RAW}

# Nhìn 20 cặp before/after. Ký hiệu cờ PHẢI còn nguyên sau khi dịch.
!python -m chessvi.data.translate --input-jsonl {RAW} --split sft --backend llm --limit 20 --dry-run

In [ ]:
# Chạy thật: bỏ --limit ở bước chuẩn hoá để lấy đủ 39.601 mẫu.
!python -m chessvi.data.c1 --split sft --out {RAW}

# --resume đọc lại checkpoint nếu Colab ngắt giữa chừng.
!python -m chessvi.data.translate --input-jsonl {RAW} --split sft --backend llm --out-dir {DRIVE_ROOT}/data/translated --resume

!ls -la {TRANSLATED}

## 6. Validate dữ liệu đã dịch (T6)

`--out-clean` ghi tập đã pass ra `data/validated/sft` — đây chính là đầu vào
của SFT. Không có bước này thì không có file nào chứa "dữ liệu đã validate".

Kỳ vọng loại 5–15%. Trên 25% script sẽ in cảnh báo to: pipeline dịch có vấn đề,
quay lại T5 chứ đừng train.

**Chốt chặn thủ công:** trước khi sang cell tiếp theo, tự đọc tay 200 mẫu ngẫu
nhiên đã pass. Không ai thay bạn làm bước này được.

In [ ]:
TRANSLATED = f"{DRIVE_ROOT}/data/translated/sft"
DATA = f"{DRIVE_ROOT}/data/validated/sft"

!python -m chessvi.data.validate \
    --input {TRANSLATED} \
    --rejected {DRIVE_ROOT}/data/rejected.jsonl \
    --out-clean {DATA}

!ls -la {DATA}

## 7. Smoke test

5 step với model 0.6B. Cell này phải chạy xong không lỗi **trước khi** tốn CU
cho lần train thật.

In [ ]:
!python -m chessvi.train.sft \
    --data {DATA} \
    --base-model Qwen/Qwen3-0.6B \
    --limit 50 --max-steps 5 \
    --output-dir /content/outputs/smoke \
    --no-push

## 8. Train thật

Đổi `HUB_MODEL_ID` thành repo của bạn. Repo được tạo ở chế độ **private**.

In [ ]:
HUB_MODEL_ID = "trantrien1/chessvi-4b-sft"
OUTPUT_DIR = f"{DRIVE_ROOT}/outputs/sft"

!python -m chessvi.train.sft \
    --data {DATA} \
    --base-model Qwen/Qwen3-4B \
    --output-dir {OUTPUT_DIR} \
    --hub-model-id {HUB_MODEL_ID} \
    --epochs 2 --batch-size 1 --grad-accum 16 --save-steps 200

## 9. Resume sau khi Colab ngắt

Chạy lại cell 1–4 rồi chạy cell này. `--resume` đọc checkpoint mới nhất trong
`--output-dir`; nếu output nằm trên Drive thì checkpoint vẫn còn nguyên.

In [ ]:
!ls -la {OUTPUT_DIR} | head -20

!python -m chessvi.train.sft \
    --data {DATA} \
    --base-model Qwen/Qwen3-4B \
    --output-dir {OUTPUT_DIR} \
    --hub-model-id {HUB_MODEL_ID} \
    --epochs 2 --batch-size 1 --grad-accum 16 --save-steps 200 \
    --resume

## 10. Chốt chặn sau T8

Accuracy trên test set phải đạt **30–38%**. Dưới 20% nghĩa là dữ liệu có vấn đề
— quay lại T5, **đừng** chạy RL.

```bash
python -m chessvi.eval.puzzle_acc --puzzles data/puzzles/test.parquet \
    --backend hf --model-path outputs/sft --out reports/sft_acc.csv
```